# 🧩 Mini-Lab: Validate at the Boundaries, Configure from Env

**Module 7: Trust but Verify** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Define** input and output schemas and reject seeded bad payloads
2. **Load** a `Settings` object from env vars + `.env`
3. **Confirm** `.env` is git-ignored and `.env.example` has no secrets

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Boundary validation | Check user input and model output — the parts you don't control |
| Layered configuration | Defaults in code, overrides in environment, secrets never in Git |


## Why Boundaries?

In an AI pipeline you control the code in the middle, but **not** the data at the edges: user input can be missing fields or have wrong types, and model output can be malformed or off-schema. If you validate **at the boundary**, the rest of your code can trust the types it receives.

## Step 1 — Define a Schema with `@dataclass` and Type Hints

We'll model a simple extraction request: a user sends text and asks the LLM to pull out entities.

In [2]:
from dataclasses import dataclass


@dataclass
class ExtractionRequest:
    """Schema for an incoming extraction request."""
    text: str
    entity_types: list[str]
    max_entities: int = 10  # sensible default

The dataclass gives us a clear **shape**: we know exactly which fields exist and what types they should be. But a dataclass alone won't reject bad data — we need a validation step.

## Step 2 — Write a Boundary Validator

We write a small `validate_request` function that checks the raw dictionary (as it would arrive from an HTTP body or JSON file) and either returns a clean `ExtractionRequest` or raises a clear error.

> 🧩 **Boundary validation.** The input and output checks from the video, now in code.

In [3]:
class ValidationError(Exception):
    """Raised when input data fails boundary validation."""


def validate_request(raw: dict) -> ExtractionRequest:
    """Validate a raw dict at the system boundary and return a typed object."""
    # --- 'text' must exist and be a non-empty string ---
    text = raw.get("text")
    if not isinstance(text, str) or not text.strip():
        raise ValidationError("'text' is required and must be a non-empty string")

    # --- 'entity_types' must exist and be a non-empty list of strings ---
    entity_types = raw.get("entity_types")
    if not isinstance(entity_types, list) or len(entity_types) == 0:
        raise ValidationError("'entity_types' is required and must be a non-empty list")
    if not all(isinstance(t, str) for t in entity_types):
        raise ValidationError("Every item in 'entity_types' must be a string")

    # --- Optional field: fall back to the default, but check the type ---
    max_entities = raw.get("max_entities", 10)
    if not isinstance(max_entities, int) or max_entities < 1:
        raise ValidationError("'max_entities' must be an integer >= 1")

    # Fail on the first problem; if we get here, the data is clean
    return ExtractionRequest(
        text=text.strip(),
        entity_types=entity_types,
        max_entities=max_entities,
    )

## Step 3 — Test with Good and Bad Payloads

We'll try one valid and one invalid payload.

In [4]:
test_payloads = [
    # ✅ Valid
    {"text": "Tesla reported $25B revenue.", "entity_types": ["organization", "money"]},
    # ❌ Missing 'text'
    {"entity_types": ["person"]},
]

for i, payload in enumerate(test_payloads, 1):
    try:
        result = validate_request(payload)
        print(f"Payload {i} ✅  → {result}")
    except ValidationError as exc:
        print(f"Payload {i} ❌  → {exc}")

Payload 1 ✅  → ExtractionRequest(text='Tesla reported $25B revenue.', entity_types=['organization', 'money'], max_entities=10)
Payload 2 ❌  → 'text' is required and must be a non-empty string


## Step 4 — Validate the Output Boundary

Model outputs are just as untrustworthy as user inputs. Let's define a schema for what we expect an LLM extraction to return, then validate simulated model output.

In [5]:
import json


@dataclass
class ExtractedEntity:
    """One entity extracted by the model."""
    name: str
    entity_type: str
    confidence: float


@dataclass
class ExtractionResponse:
    """Schema for the model's extraction output."""
    entities: list[ExtractedEntity]


def validate_response(raw_json: str) -> ExtractionResponse:
    """Validate model output at the output boundary."""
    # --- Parse JSON: models sometimes return broken JSON ---
    try:
        data = json.loads(raw_json)
    except json.JSONDecodeError as exc:
        raise ValidationError(f"Model returned invalid JSON: {exc}")

    if not isinstance(data.get("entities"), list):
        raise ValidationError("Model output missing 'entities' list")

    # --- Check each entity, raising on the first bad field ---
    validated: list[ExtractedEntity] = []
    for item in data["entities"]:
        name = item.get("name")
        if not isinstance(name, str) or not name.strip():
            raise ValidationError("Entity 'name' must be a non-empty string")
        if not isinstance(item.get("entity_type"), str):
            raise ValidationError("Entity 'entity_type' must be a string")
        conf = item.get("confidence")
        if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
            raise ValidationError("Entity 'confidence' must be a number between 0 and 1")
        validated.append(ExtractedEntity(
            name=name.strip(),
            entity_type=item["entity_type"],
            confidence=float(conf),
        ))

    return ExtractionResponse(entities=validated)

In [6]:
# Simulated model outputs — one good, one bad
model_outputs = [
    # ✅ Well-formed
    '{"entities": [{"name": "OpenAI", "entity_type": "organization", "confidence": 0.95}]}',
    # ❌ confidence out of range
    '{"entities": [{"name": "Paris", "entity_type": "location", "confidence": 1.5}]}',
]

for i, output in enumerate(model_outputs, 1):
    try:
        result = validate_response(output)
        print(f"Output {i} ✅  → {result}")
    except ValidationError as exc:
        print(f"Output {i} ❌  → {exc}")

Output 1 ✅  → ExtractionResponse(entities=[ExtractedEntity(name='OpenAI', entity_type='organization', confidence=0.95)])
Output 2 ❌  → Entity 'confidence' must be a number between 0 and 1


In [7]:
validate_response('{"entities": [{"name": "Mars", "entity_type": "planet", "confidence": 0.88}]}')

ExtractionResponse(entities=[ExtractedEntity(name='Mars', entity_type='planet', confidence=0.88)])

---
## Settings from the environment

Same sitting: validate IO, then load config from env.


## Why Layered Configuration?

Settings come from three sources, each overriding the last:

```
Layer 1 (lowest priority):  Hard-coded DEFAULTS in code
Layer 2:                    .env file  (local dev, git-ignored)
Layer 3 (highest priority): Real ENVIRONMENT VARIABLES  (CI/CD, containers)
```

You always have a working default, developers customize via `.env`, and production overrides via real environment variables win. Secrets live in `.env` or the environment, never in Git.

## Step 1 — Create a `.env` File

In a real project you'd create `.env` by hand in your project root. Here we'll write one programmatically so the lab is self-contained.

> **Key rule:** This file is for *local development only*. It must **never** be committed to version control.

> 🧩 **Configuration layers.** The defaults, environment and secrets layering from the video, built here.

In [8]:
from pathlib import Path

env_content = """# Local development settings - DO NOT COMMIT
OPENAI_API_KEY=sk-fake-dev-key-12345
MODEL_NAME=gpt-4o-mini
REQUEST_TIMEOUT=45
"""

Path(".env").write_text(env_content.strip(), encoding="utf-8")
print("Wrote .env (demo)")

Wrote .env (demo)


## Step 2 — Add `.env` to `.gitignore`

Add `.env` to `.gitignore` before you put secrets in it, so a careless `git add .` can't commit it. If you share a `.env.example` to show teammates which keys exist, it must hold only placeholder values, never real secrets.

In [9]:
gitignore_path = Path(".gitignore")

# Read existing content (or start empty)
existing = gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else ""

# Append the rule if it's missing
if ".env" not in existing.splitlines():
    with open(gitignore_path, "a", encoding="utf-8") as f:
        f.write("\n.env\n")

print("✅ .env is in .gitignore")

✅ .env is in .gitignore


## Step 3 — Build the Layered Settings Object

We define a `Settings` dataclass with sensible defaults (Layer 1), then load values from the `.env` file (Layer 2) using `python-dotenv` (`pip install python-dotenv`), and finally allow real environment variables to override everything (Layer 3).

The function `load_settings()` implements all three layers in order.

In [10]:
import os
from dataclasses import dataclass
from dotenv import load_dotenv


@dataclass
class Settings:
    """Application settings built from layered configuration."""
    openai_api_key: str
    model_name: str
    request_timeout: int


# --- Layer 1: Hard-coded defaults ---
DEFAULTS = {
    "OPENAI_API_KEY": "",          # no default — must be provided
    "MODEL_NAME": "gpt-4o-mini",
    "REQUEST_TIMEOUT": "30",        # env vars are always strings
}


def load_settings(env_path: str = ".env") -> Settings:
    """Load settings: defaults → .env file → environment variables."""

    # --- Layer 2: .env file (does NOT override already-set env vars) ---
    load_dotenv(env_path, override=False, encoding="utf-8")

    # --- Layer 3: os.environ (real env vars) wins, falling back to defaults ---
    api_key = os.environ.get("OPENAI_API_KEY", DEFAULTS["OPENAI_API_KEY"])
    if not api_key:
        raise ValueError("OPENAI_API_KEY is required but not set in .env or environment")

    return Settings(
        openai_api_key=api_key,
        model_name=os.environ.get("MODEL_NAME", DEFAULTS["MODEL_NAME"]),
        request_timeout=int(os.environ.get("REQUEST_TIMEOUT", DEFAULTS["REQUEST_TIMEOUT"])),
    )

In [11]:
# Demo only: drop any real OPENAI_API_KEY so this prints the fake key from .env
os.environ.pop("OPENAI_API_KEY", None)

settings = load_settings()
print(settings)

Settings(openai_api_key='sk-fake-dev-key-12345', model_name='gpt-4o-mini', request_timeout=45)


In [12]:
print(f"Key ends with: ...{settings.openai_api_key[-5:]}")
print(f"Timeout: {settings.request_timeout}s (from .env, overrode default of 30)")

Key ends with: ...12345
Timeout: 45s (from .env, overrode default of 30)


## Step 4 — Layer 3 Overrides Everything

Real environment variables (Layer 3) take the highest priority. Let's simulate setting one and see it override the `.env` value.

In [13]:
# Simulate a real env var (as CI/CD or Docker would set it)
os.environ["REQUEST_TIMEOUT"] = "120"

settings = load_settings()
print(f"request_timeout = {settings.request_timeout}")  # 120, not 45 from .env

# Clean up so it doesn't affect later cells
del os.environ["REQUEST_TIMEOUT"]

request_timeout = 120


## Step 5 — Fail Loudly on Missing Secrets

Verify that `load_settings()` fails at startup if `OPENAI_API_KEY` is missing.

In [14]:
# Write a .env without the API key
Path(".env").write_text("MODEL_NAME=gpt-4o-mini\n", encoding="utf-8")

# Remove from environment too, so no layer provides it
os.environ.pop("OPENAI_API_KEY", None)

try:
    bad_settings = load_settings()
except ValueError as exc:
    print(f"✅ Caught expected error: {exc}")

# Restore the original .env (write_text returns a char count; assign it so it isn't displayed)
_ = Path(".env").write_text(env_content.strip(), encoding="utf-8")

✅ Caught expected error: OPENAI_API_KEY is required but not set in .env or environment


In [15]:
# Remove the demo files so they don't linger
Path(".env").unlink(missing_ok=True)
print("Removed .env demo file")

Removed .env demo file


## 🎯 Summary

### Key Takeaways

1. **Boundary validation** — check data where it enters (user input) and where it leaves (model output) your system, so inner code can trust it
2. **Dataclass schemas** — `@dataclass` declares the exact shape (fields and defaults) you expect, serving as living documentation
3. **Type hints** — annotating fields with `str`, `int`, `list[str]`, `float` makes expectations explicit and enables runtime checks
4. **Fail loudly** — a clear `ValidationError` at the boundary is far easier to debug than a mysterious `KeyError` deep inside your pipeline